In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train_full = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_train_full_clean.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_test_future.parquet')

print(train_full.shape, test.shape)

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

(749609, 34) (114111, 34)


In [2]:
# ============================================================
# 0. 라이브러리 불러오기
# ============================================================

import numpy as np

from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [3]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [4]:
# ============================================================
# 2. 종속변수 설정
# ============================================================

y_train_full = train_full[target_col]
y_test = test[target_col]

In [5]:
# ============================================================
# 3. station_id 원핫인코딩 독립변수 생성
# ============================================================

# OHE 전 독립변수 데이터셋
X_train_full = train_full[num_cols_fe]
X_test = test[num_cols_fe]

# 원핫인코딩
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 X
    sparse_output=True  # sparse: 희소한, 희소행렬(0이 아닌 값이 드문 행렬)
    # 원핫인코딩 결과를 전체 다 저장 X, 희소한 1의 값의 위치만 저장(메모리 사용 적음)
)

# 학습은 train만
ohe.fit(train_full[['station_id']])  # sklearn 인코더는 2차원 행렬을 받는다.

# 변환
train_st_id_ohe = ohe.transform(train_full[['station_id']])
test_st_id_ohe = ohe.transform(test[['station_id']])
# 여기까지는 일반적은 인코딩 과정

# train, val 셋에 합치기
from scipy.sparse import hstack
# scipy안에 희소행렬을 다루는 서브캐키지 scipy.sparse
# hstack(horizontal stack)은 concat(axis=1)과 같은 역할, df가 아닌 행렬도 붙일 수 있는 함수

X_train_with_station_ohe = hstack([
    X_train_full,
    train_st_id_ohe
]).tocsr()  # df + 희소행렬(ohe) -> csr형태로 저장하여 sparse 상태 유지
# 이 변수는 더이상 df가 아님

X_test_with_station_ohe = hstack([
    X_test,
    test_st_id_ohe
]).tocsr()

print("원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 들어남")
print(X_train_full.shape, X_train_with_station_ohe.shape)
print(X_test.shape, X_test_with_station_ohe.shape)
# 제공된 데이터셋의 관측소 개수는 465개(md파일에 명시)
# 기존컬럼 31개 + 465 = 496이 정상

원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 들어남
(749609, 31) (749609, 496)
(114111, 31) (114111, 496)


In [ ]:
# ============================================================
# 4. 모델15 (최종검증: 기존 catboost모델, y_test)
# ============================================================
# train_test 분포비교에서 이상치 제거가 한번 있었음, 이것 반영해서 기본모델 한번 돌려보기

# CatBoost 전용 데이터셋
X_train_with_station = train_full[num_cols_fe + ['station_id']].copy()
X_test_with_station = test[num_cols_fe + ['station_id']].copy()

X_train_with_station['station_id'] = X_train_with_station['station_id'].astype(str)
X_test_with_station['station_id'] = X_test_with_station['station_id'].astype(str)

# 모델 정의
model_base_no_ohe = CatBoostRegressor(
    random_state=42,
    iterations=3000,
    cat_features=['station_id']
)

# 학습
model_base_no_ohe.fit(
    X_train_with_station,
    y_train_full,
    early_stopping_rounds=100,
    eval_set=(X_test_with_station, y_test),
    verbose=200
)

# 예측
pred_test_no_ohe = model_base_no_ohe.predict(X_test_with_station)
pred_train_no_ohe = model_base_no_ohe.predict(X_train_with_station)

# 평가
rmse_test_no_ohe = root_mean_squared_error(y_test, pred_test_no_ohe)
rmse_train_no_ohe = root_mean_squared_error(y_train_full, pred_train_no_ohe)

print('Best iteration:', model_base_no_ohe.best_iteration_)
print('Tree count:', model_base_no_ohe.tree_count_)
print('Test RMSE:', rmse_test_no_ohe)
print('Train RMSE:', rmse_train_no_ohe)
print('Gap:', rmse_test_no_ohe - rmse_train_no_ohe)

Learning rate set to 0.073655
0:	learn: 0.3815887	test: 0.3705444	best: 0.3705444 (0)	total: 111ms	remaining: 5m 32s
200:	learn: 0.2245861	test: 0.2176650	best: 0.2176650 (200)	total: 20.7s	remaining: 4m 48s
400:	learn: 0.2136080	test: 0.2114690	best: 0.2114690 (400)	total: 40.7s	remaining: 4m 23s
600:	learn: 0.2069185	test: 0.2087125	best: 0.2087125 (600)	total: 1m 1s	remaining: 4m 4s
800:	learn: 0.2019976	test: 0.2074117	best: 0.2073848 (791)	total: 1m 21s	remaining: 3m 44s
1000:	learn: 0.1980992	test: 0.2066846	best: 0.2066724 (984)	total: 1m 41s	remaining: 3m 22s
1200:	learn: 0.1949022	test: 0.2060690	best: 0.2060264 (1190)	total: 2m 1s	remaining: 3m 1s
1400:	learn: 0.1921277	test: 0.2058874	best: 0.2058632 (1392)	total: 2m 21s	remaining: 2m 41s
1600:	learn: 0.1894397	test: 0.2055767	best: 0.2055767 (1600)	total: 2m 41s	remaining: 2m 20s
1800:	learn: 0.1871858	test: 0.2052672	best: 0.2052343 (1749)	total: 3m 1s	remaining: 2m
2000:	learn: 0.1850077	test: 0.2050841	best: 0.2050696 (1

In [ ]:
# ============================================================
# 4. 모델15-1 (CatBoost, iteration 고정)
# ============================================================
# validation 예측 iteration 1779 고정

model_base_no_ohe_fixed = CatBoostRegressor(
    random_state=42,
    iterations=1779,
    cat_features=['station_id'],
    learning_rate=0.073655,
    verbose=200
)

model_base_no_ohe_fixed.fit(
    X_train_with_station,
    y_train_full
)

pred_test_fixed = model_base_no_ohe_fixed.predict(X_test_with_station)
pred_train_fixed = model_base_no_ohe_fixed.predict(X_train_with_station)

rmse_test_fixed = root_mean_squared_error(y_test, pred_test_fixed)
rmse_train_fixed = root_mean_squared_error(y_train_full, pred_train_fixed)

print('Test RMSE:', rmse_test_fixed)
print('Train RMSE:', rmse_train_fixed)
print('Gap:', rmse_test_fixed - rmse_train_fixed)

0:	learn: 0.3815887	total: 107ms	remaining: 3m 10s
200:	learn: 0.2245861	total: 22.4s	remaining: 2m 55s
400:	learn: 0.2136080	total: 46.2s	remaining: 2m 38s
600:	learn: 0.2069185	total: 1m 7s	remaining: 2m 11s
800:	learn: 0.2019976	total: 1m 30s	remaining: 1m 50s
1000:	learn: 0.1980992	total: 1m 53s	remaining: 1m 28s
1200:	learn: 0.1949022	total: 2m 13s	remaining: 1m 4s
1400:	learn: 0.1921277	total: 2m 35s	remaining: 41.9s
1600:	learn: 0.1894397	total: 2m 57s	remaining: 19.8s
1778:	learn: 0.1873955	total: 3m 18s	remaining: 0us
Test RMSE: 0.20526333315501474
Train RMSE: 0.18651485041480456
Gap: 0.01874848274021018


In [13]:
# ============================================================
# 6. XGBoost (Test 검증)
# ============================================================

from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error
import time

# 모델 정의
model_xgb = XGBRegressor(
    n_estimators=3000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    eval_metric='rmse',
    tree_method='hist',
    early_stopping_rounds=100,
    n_jobs=2,
    random_state=42
)

start = time.time()

# 학습
model_xgb.fit(
    X_train_with_station_ohe,
    y_train_full,
    eval_set=[(X_test_with_station_ohe, y_test)],
    verbose=200
)

# 예측
pred_test_xgb = model_xgb.predict(X_test_with_station_ohe)
pred_train_xgb = model_xgb.predict(X_train_with_station_ohe)

# 평가
rmse_test_xgb = root_mean_squared_error(y_test, pred_test_xgb)
rmse_train_xgb = root_mean_squared_error(y_train_full, pred_train_xgb)

print('Best iteration:', model_xgb.best_iteration)
print('Test RMSE:', rmse_test_xgb)
print('Train RMSE:', rmse_train_xgb)
print('Gap:', rmse_test_xgb - rmse_train_xgb)
print('Time:', round(time.time() - start, 1), 'sec')

[0]	validation_0-rmse:0.37434
[200]	validation_0-rmse:0.21492
[400]	validation_0-rmse:0.21146
[600]	validation_0-rmse:0.20953
[800]	validation_0-rmse:0.20823
[1000]	validation_0-rmse:0.20742
[1200]	validation_0-rmse:0.20687
[1400]	validation_0-rmse:0.20642
[1600]	validation_0-rmse:0.20632
[1800]	validation_0-rmse:0.20620
[2000]	validation_0-rmse:0.20612
[2200]	validation_0-rmse:0.20597
[2372]	validation_0-rmse:0.20599
Best iteration: 2272
Test RMSE: 0.2059450813794177
Train RMSE: 0.15293391723363373
Gap: 0.05301116414578397
Time: 1499.8 sec


In [6]:
# ============================================================
# 5. 모델12-8 RF8 (최종 Test 검증)
# ============================================================

# from sklearn.ensemble import RandomForestRegressor
# from sklearn.metrics import root_mean_squared_error
import time

# 모델 정의
models_bag = {
    'RandomForest_8': RandomForestRegressor(
        n_estimators=30,
        max_depth=20,
        max_features=0.3,
        min_samples_leaf=10,
        n_jobs=2,
        verbose=10,
        random_state=42
    )
}

for name, model in models_bag.items():

    start = time.time()

    # 학습
    model.fit(
        X_train_with_station_ohe,
        y_train_full
    )

    # 예측
    pred = model.predict(X_test_with_station_ohe)
    pred_train = model.predict(X_train_with_station_ohe)

    # RMSE
    rmse_test = root_mean_squared_error(y_test, pred)
    rmse_train = root_mean_squared_error(y_train_full, pred_train)

    print('Model:', name)
    print('Test RMSE:', rmse_test)
    print('Train RMSE:', rmse_train)
    print('Gap:', rmse_test - rmse_train)
    print('Time:', round(time.time() - start, 1), 'sec')

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.


building tree 1 of 30
building tree 2 of 30
building tree 3 of 30


[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:   18.8s


building tree 4 of 30
building tree 5 of 30
building tree 6 of 30


[Parallel(n_jobs=2)]: Done   4 tasks      | elapsed:   47.1s


...

[Parallel(n_jobs=2)]: Done  21 tasks      | elapsed:  4.2min


building tree 24 of 30
building tree 25 of 30
building tree 26 of 30
building tree 27 of 30
building tree 28 of 30
building tree 29 of 30
building tree 30 of 30


[Parallel(n_jobs=2)]: Done  30 out of  30 | elapsed:  5.9min finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done   4 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done   9 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done  14 tasks      | elapsed:    0.1s
[Parallel(n_jobs=2)]: Done  21 tasks      | elapsed:    0.2s
[Parallel(n_jobs=2)]: Done  30 out of  30 | elapsed:    0.3s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:    0.1s
[Parallel(n_jobs=2)]: Done   4 tasks      | elapsed:    0.2s
[Parallel(n_jobs=2)]: Done   9 tasks      | elapsed:    0.7s
[Parallel(n_jobs=2)]: Done  14 tasks      | elapsed:    1.1s
[Parallel(n_jobs=2)]: Done  21 tasks      | elapsed:    1.7s


Model: RandomForest_8
Test RMSE: 0.21491046698802127
Train RMSE: 0.19123540196908498
Gap: 0.023675065018936292
Time: 357.9 sec


[Parallel(n_jobs=2)]: Done  30 out of  30 | elapsed:    2.4s finished
